# Session 3: Introduction to Image Processing Techniques and Image Restoration

We will cover the following topics in this tutorial:
- Image visualization and histogram visualization
- Histogram matching and equalization
- Denoising
- Spatial filtering

**Learning objectives** — by the end of this notebook you should be able to:
- Compute and interpret an image histogram and its cumulative distribution function (CDF)
- Implement histogram equalization and histogram matching from scratch
- Add and characterize common noise models (Gaussian, salt-and-pepper)
- Denoise images with linear (Gaussian) and non-linear (median) spatial filters, and explain when each is appropriate

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

np.random.seed(0)

# Read the test image
image = cv2.imread("data/test1.jpg")

# Convert the image from BGR to RGB
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Visualize the image
plt.imshow(image)
plt.axis("off")
plt.show()


# Ex 1 (a): Plot the image histogram
# define a function: plot_image_histogram(image)
# which will be used in the next exercises
# TODO: your code here


plot_image_histogram(image)


# Ex 1 (b): Plot the image normalized cumulative histogram
# define a function: plot_image_cdf(image)
# which will be used in the next exercises
# TODO: your code here


plot_image_cdf(image), plt.title("Image Normalized CDF")

## Let's fix the image dynamic range by performing histogram equalization

In [ ]:
import numpy as np


# Ex 2: Implement image equalization (only using numpy)
# define a method called histogram_equalization(image)
# TODO: your code here


# Apply histogram equalization to the image
equalized_image = histogram_equalization(image)

# Visualize the equalized image
plt.imshow(equalized_image)
plt.axis("off")
plt.show()
plot_image_histogram(equalized_image)

Note: `histogram_equalization` above equalizes a single, joint RGB histogram (all channels pooled together), which is simple but can shift hues — equalizing each channel independently, or only the luminance channel (e.g., V in HSV / Y in YCrCb), preserves hue better at the cost of a less aggressive contrast boost.

### Let's compare the cumulative distribution of these two images

**Ex 3:** Explain why the equalized image's CDF looks the way it does, compared to the original.

In [ ]:
plt.plot(np.cumsum(np.histogram(image.ravel(), bins=256, range=[0, 256])[0]))
plt.plot(np.cumsum(np.histogram(equalized_image.ravel(), bins=256, range=[0, 256])[0]))
plt.legend(["Original", "Equalized"])

## Histogram matching

We will match the histogram of one image into another. 

In [ ]:
# We will get two test images from skimage library
from skimage import data

img1 = getattr(data, "cat")()
img2 = getattr(data, "coffee")()
plt.subplot(1, 2, 1)
plt.imshow(img1)
plt.axis("off"), plt.title("img1: Cat")
plt.subplot(1, 2, 2)
plt.imshow(img2)
plt.axis("off"), plt.title("img2: Coffee")

In [ ]:
import numpy as np


# Ex 4: Implement histogram matching (only using numpy)
# define a method called histogram_matching(image, target_image)
# TODO: your code here

In [ ]:
# Let's visualize the results with the method you defined above
img1_matched = histogram_matching(img1, img2)
plt.figure(figsize=(15, 10))
plt.subplot(2, 3, 1), plt.imshow(img1), plt.axis("off"), plt.title("(Original)")
plt.subplot(2, 3, 2), plt.imshow(img1_matched), plt.axis("off"), plt.title("(Matched)")
plt.subplot(2, 3, 3), plt.imshow(img2), plt.axis("off"), plt.title("(Target)")
plt.subplot(2, 3, 4), plot_image_cdf(img1), plt.title("CDF (Original)"),
plt.subplot(2, 3, 5), plot_image_cdf(img1_matched), plt.title("CDF (Matched)")
plt.subplot(2, 3, 6), plot_image_cdf(img2), plt.title("CDF (Target)")

## Introduction to spatial filtering and image denoising

### Adding noise to clean images

In [ ]:
from skimage import data

# Let's use a test image from skimage library
img = getattr(data, "cat")()
plt.imshow(img), plt.axis("off"), plt.title("Original Image")

In [ ]:
# Ex 5 (a): Using only numpy, create a function
# capable of adding Gaussian noise to an image
# define a method called
# add_gaussian_noise(image, mean, std) -> image
# TODO: your code here


# Ex 5 (b): Using only numpy, create a function
# capable of adding Salt and Pepper noise to an image
# define a method called add_salt_and_pepper_noise(image, amount)
# amount is the percentage of individual channel values to corrupt (we sample flat indices
# into the raveled RGB array, so a given pixel may have only one or two of its channels
# flipped to salt/pepper, not necessarily all three)
# add_salt_and_pepper_noise(image, amount) -> image
# TODO: your code here

In [ ]:
# Let's visualize the results for gaussian noise of different standard deviations.
plt.figure(figsize=(20, 15))
sigma = np.arange(0, 100, 30)
n = len(sigma)
for i, s in enumerate(sigma):
    plt.subplot(1, n, i + 1), plt.imshow(add_gaussian_noise(img, 0, s)), plt.axis("off"), plt.title(
        f"Gaussian Noise (sigma={s})"
    )

In [ ]:
# Let's visualize the results for salt and pepper noise of different amounts.
plt.figure(figsize=(20, 15))
amounts = np.arange(0, 100, 30)
n = len(amounts)
for i, a in enumerate(amounts):
    plt.subplot(1, n, i + 1), plt.imshow(add_salt_and_pepper_noise(img, a)), plt.axis("off"), plt.title(
        f"Salt and Pepper (amount={a})"
    )

### Image denoising

In [ ]:
# Ex 6: Remove noise by using a Gaussian filter
# implement a method called
# remove_noise(image, sigma) -> image
# you can use opencv (tip: cv2.GaussianBlur)
# Bonus, you can try to implement this from scratch (using numpy)
# TODO: your code here

In [ ]:
from skimage import data

clean_img = getattr(data, "cat")()
noisy_img = add_gaussian_noise(clean_img, 0, 30)
sigma = 10
denoised_img = remove_noise(noisy_img, sigma)

plt.figure(figsize=(15, 10))
plt.subplot(1, 3, 1), plt.imshow(clean_img), plt.axis("off"), plt.title("Original")
plt.subplot(1, 3, 2), plt.imshow(noisy_img), plt.axis("off"), plt.title("Noisy")
plt.subplot(1, 3, 3), plt.imshow(denoised_img), plt.axis("off"), plt.title("Denoised")

Plot the MSE for a given level of noise (fixed noise sigma) for different kernel widths. 

In [ ]:
clean_img = getattr(data, "cat")()  # get a clean image
noisy_img = add_gaussian_noise(clean_img, 0, 30)  # compute a noisy image for a fixed sigma

sigmas = np.arange(0.5, 30, 1)  # sigmas to test for denoising (start > 0 so the filter does something)


# Ex 7: Check the denoising MSE for different kernel sigmas
# TODO: your code here

plt.plot(sigmas, mse_values), plt.xlabel("Sigma"), plt.ylabel("MSE"), plt.title("MSE vs Sigma")

## Denoising salt-and-pepper noise

Gaussian filtering assumes noise that varies smoothly (like Gaussian noise) — averaging neighboring pixels cancels it out. Salt-and-pepper noise is different: a small fraction of pixels are replaced by extreme outliers (0 or 255), and a linear average gets dragged toward those outliers. A **median filter**, which replaces each pixel with the median of its neighborhood, is much more robust to this kind of noise because the median discards outliers instead of averaging them in.

**Ex 8:** Denoise the salt-and-pepper image below with (a) the Gaussian filter from Ex 6 and (b) a median filter (`cv2.medianBlur` or `scipy.ndimage.median_filter`). Compare the results visually and with MSE, and explain why the median filter wins.

In [ ]:
clean_img = getattr(data, "cat")()
sp_img = add_salt_and_pepper_noise(clean_img, 10)


# TODO: your code here

plt.figure(figsize=(20, 5))
plt.subplot(1, 4, 1), plt.imshow(clean_img), plt.axis("off"), plt.title("Original")
plt.subplot(1, 4, 2), plt.imshow(sp_img), plt.axis("off"), plt.title("Salt & pepper")
plt.subplot(1, 4, 3), plt.imshow(gaussian_denoised), plt.axis("off"), plt.title(f"Gaussian (MSE={mse_gaussian:.1f})")
plt.subplot(1, 4, 4), plt.imshow(median_denoised), plt.axis("off"), plt.title(f"Median (MSE={mse_median:.1f})")
plt.show()

print(f"MSE Gaussian filter: {mse_gaussian:.2f}")
print(f"MSE Median filter:   {mse_median:.2f}")
assert mse_median < mse_gaussian, "Median filter should beat Gaussian on salt-and-pepper noise"